In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


**导入库并读取数据**

In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# 添加随机森林模型与交叉验证
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

train = pd.read_csv('/kaggle/input/titanic/train.csv')
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [3]:
test = pd.read_csv('/kaggle/input/titanic/test.csv')
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.1+ KB


**数据预处理**

In [4]:
# 用中位数填充Age
train['Age'] = train['Age'].fillna(train['Age'].median())
test['Age'] = test['Age'].fillna(train['Age'].median())

# 用众数填充Embarked
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

# 用中位数填充Fare
test['Fare'] = test['Fare'].fillna(test['Fare'].median())

# 将Sex和Embarked转换成数字
le = LabelEncoder()
train['Sex'] = le.fit_transform(train['Sex'])
test['Sex'] = le.transform(test['Sex'])

train['Embarked'] = le.fit_transform(train['Embarked'])
test['Embarked'] = le.transform(test['Embarked'])

**特征选择和模型训练**

In [5]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = train[features]
y = train['Survived']

# 逻辑回归模型
model = LogisticRegression(max_iter=200)
model.fit(X, y)

# 随机森林模型
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 交叉验证评估模型
cv_scores = cross_val_score(rf_model, X, y, cv=5)

# 输出每折得分
print('交叉验证得分：', cv_scores) 
print('平均交叉验证得分：', cv_scores.mean())

rf_model.fit(X, y)

交叉验证得分： [0.77094972 0.81460674 0.86516854 0.7752809  0.82022472]
平均交叉验证得分： 0.8092461239093591


RandomForestClassifier(random_state=42)

**预测与生成提交文件**

In [6]:
X_test = test[features]

# 逻辑回归预测
predictions = model.predict(X_test)

# 随机森林预测
predictions_rf = rf_model.predict(X_test)

# 提交逻辑回归预测结果
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

# 提交随机森林预测结果
submission_rf = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions_rf
})

submission.to_csv('/kaggle/working/submission.csv', index=False)    # 逻辑回归结果
submission_rf.to_csv('/kaggle/working/titanic_rf_submission.csv', index=False)    # 随机森林结果
print('提交文件已生成')

print('逻辑回归预测幸存人数:', sum(predictions))
print('随机森林预测幸存人数:', sum(predictions_rf))

提交文件已生成
逻辑回归预测幸存人数: 155
随机森林预测幸存人数: 151
